# Feature Extraction Pipeline — Test Notebook
Tests each stage of `semantic_structure/extractor.py`:
1. HTML parsing → `(token, tag)` pairs
2. Vocab building + GloVe loading
3. Ω' matrix construction

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import config
import rust_extractor
from semantic_structure.db import load_records
from semantic_structure.extractor import (
    _parse_html,
    _tokenize,
    build_corpus_vocab,
    extract_page,
)

print('Imports OK')
print(f'DB path:    {config.DB_PATH}')
print(f'GloVe path: {config.GLOVE_PATH}')
print(f'M={config.M}, N={config.N}')

Imports OK
DB path:    /Users/umair/Library/Application Support/mlops/pages.db
GloVe path: /Users/umair/mlops/models/semantic_structure/GloVe/dolma_300_2024_1.2M.100_combined.txt
M=512, N=16


## 1. Tokenizer

In [2]:
cases = [
    ("Hello, World!",               ["hello", "world"]),
    ("PyTorch 2.0 docs",            ["pytorch", "2.0", "docs"]),
    ("don't stop",                  ["don't", "stop"]),
    ("multi-head attention",        ["multi-head", "attention"]),
    ("   extra   whitespace   ",    ["extra", "whitespace"]),
    ("",                            []),
]

all_passed = True
for text, expected in cases:
    result = _tokenize(text)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_passed = False
    print(f"{status}  input={repr(text)!r:40s}  got={result}")

print()
print("All tokenizer tests passed" if all_passed else "SOME TESTS FAILED")

PASS  input="'Hello, World!'"                         got=['hello', 'world']
FAIL  input="'PyTorch 2.0 docs'"                      got=['pytorch', '2', '0', 'docs']
PASS  input='"don\'t stop"'                           got=["don't", 'stop']
PASS  input="'multi-head attention'"                  got=['multi-head', 'attention']
PASS  input="'   extra   whitespace   '"              got=['extra', 'whitespace']
PASS  input="''"                                      got=[]

SOME TESTS FAILED


## 2. Database — load records

In [3]:
records = load_records(config.DB_PATH)

print(f'Total records: {len(records)}')
print()

from collections import Counter
label_counts = Counter(label for _, _, label in records)
for label, count in sorted(label_counts.items()):
    print(f'  {label:12s}: {count}')

print()
print('Sample records:')
for url, html_path, label in records[:5]:
    import os
    exists = os.path.exists(html_path)
    print(f'  [{label}] {url}')
    print(f'         html exists={exists}, path={html_path}')

Total records: 20

  productive  : 14
  skip        : 1
  waste       : 5

Sample records:
  [waste] pokemon.com/us
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/04a156a4652be738cdb86bda48b2782d07243738f38e54b61b3fd63d7787e6b2/page.html
  [waste] minecraft.net/en-us
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/fe82e14a01d19579f7d2e1292749c57a417948eb9db1ed9d7ce4c605592bd59e/page.html
  [waste] reddit.com/r/playboicarti/hot/
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/f3062ed76a5d650ff3f49b4cc5b2bf7cd4f3f31a861a8bda582d00eb316f71d2/page.html
  [waste] reddit.com/r/ChainsawMan/
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/b68022bc82a9ec114393f681214a2bf46e83365a45702a83610be5beb03fe525/page.html
  [productive] docs.pytorch.org/docs/stable/index.html
         html exists=True, path=/Users/umair/Library/Application Suppo

## 3. HTML Parser — `(token, tag)` pairs

In [4]:
from collections import defaultdict

print(f'Parsing all {len(records)} pages with M={config.M}...\n')

for url, html_path, label in records:
    pairs    = _parse_html(html_path, config.M)
    rs_pairs = rust_extractor.parse_html_py(html_path, config.M)
    tag_counts = defaultdict(int)
    for _, tag in pairs:
        tag_counts[tag] += 1
    tags_str  = ', '.join(f'{t}:{n}' for t, n in sorted(tag_counts.items()))
    rs_status = 'OK' if len(pairs) == len(rs_pairs) else 'MISMATCH'
    print(f'[{label:10s}] {url}')
    print(f'             py={len(pairs):4d}  rs={len(rs_pairs):4d} [{rs_status}]  tags: {tags_str}')

Parsing all 20 pages with M=512...

[waste     ] pokemon.com/us
             py= 512  rs= 512 [OK]  tags: em:25, h2:8, h3:102, h5:17, meta_desc:53, p:151, span:149, title:7
[waste     ] minecraft.net/en-us
             py= 512  rs= 512 [OK]  tags: h2:3, h3:15, meta_desc:79, p:293, span:115, title:7
[waste     ] reddit.com/r/playboicarti/hot/
             py= 435  rs= 435 [OK]  tags: h1:2, h2:18, p:321, span:92, title:2
[waste     ] reddit.com/r/ChainsawMan/
             py= 512  rs= 512 [OK]  tags: h1:2, h2:50, p:315, span:143, title:2
[productive] docs.pytorch.org/docs/stable/index.html
             py= 512  rs= 512 [OK]  tags: h1:2, h2:6, p:246, span:237, strong:7, title:14
[productive] chatgpt.com/
             py=  51  rs=  51 [OK]  tags: meta_desc:34, span:16, title:1
[skip      ] youtube.com/
             py= 512  rs= 512 [OK]  tags: meta_desc:31, span:480, title:1
[productive] developer.apple.com/documentation/swift/
             py= 512  rs= 512 [OK]  tags: h2:1, h3:2, meta_des

In [5]:
# Inspect first 20 (token, tag) pairs from one page
url, html_path, label = records[4]  # change index to inspect a different page
pairs = _parse_html(html_path, config.M)

print(f'Page: {url}  [{label}]')
print(f'Total pairs: {len(pairs)}')
print()
print('First 20 (token, tag) pairs:')
for tok, tag in pairs[:20]:
    print(f'  {tag:12s}  "{tok}"')

Page: docs.pytorch.org/docs/stable/index.html  [productive]
Total pairs: 512

First 20 (token, tag) pairs:
  title         "pytorch"
  title         "documentation"
  title         "pytorch"
  title         "2"
  title         "10"
  title         "documentation"
  span          "opens"
  span          "in"
  span          "a"
  span          "new"
  span          "window"
  span          "opens"
  span          "an"
  span          "external"
  span          "website"
  span          "opens"
  span          "an"
  span          "external"
  span          "website"
  span          "in"


## 4. Vocab + GloVe — `build_corpus_vocab`

In [6]:
token2idx, word_matrix, struct_matrix = build_corpus_vocab(
    records, config.GLOVE_PATH, config.N, config.M
)

print(f'Vocab size:           {len(token2idx)}')
print(f'word_matrix shape:    {word_matrix.shape}   (expected [V, k])')
print(f'struct_matrix shape:  {struct_matrix.shape}  (expected [{config.NUM_TAGS+1}, {config.N}])')
print()

# PAD row must be all zeros
assert word_matrix[0].sum() == 0,   'FAIL: PAD word row is not zero'
assert struct_matrix[0].sum() == 0, 'FAIL: PAD struct row is not zero'
print('PAD rows (index 0): all zeros — OK')

# UNK row must be non-zero (mean of known GloVe vecs)
assert word_matrix[1].sum() != 0, 'FAIL: UNK word row is zero (no GloVe hits?)'
print('UNK row  (index 1): non-zero mean — OK')

Parsing HTML corpus ...


100%|██████████| 20/20 [00:00<00:00, 72.67page/s]


Loading GloVe from /Users/umair/mlops/models/semantic_structure/GloVe/dolma_300_2024_1.2M.100_combined.txt ...


1200000 lines [00:09, 121931.41 lines/s]

GloVe dimension: 300
Vocab: 2468 tokens | GloVe coverage: 2173/2466 (88.1%)
Vocab size:           2468
word_matrix shape:    (2468, 300)   (expected [V, k])
struct_matrix shape:  (13, 16)  (expected [13, 16])

PAD rows (index 0): all zeros — OK
UNK row  (index 1): non-zero mean — OK


In [7]:
# Check a few known words
probe_words = ['pytorch', 'documentation', 'pokemon', 'reddit', 'attention', 'zzzzunknownword']
print(f'{"word":20s}  {"in vocab":10s}  {"GloVe hit":10s}  vec[:4]')
print('-' * 70)
for word in probe_words:
    idx = token2idx.get(word)
    in_vocab = idx is not None
    if idx is not None:
        vec = word_matrix[idx]
        glove_hit = vec.sum() != 0
        preview = str(vec[:4].round(3))
    else:
        glove_hit = False
        preview = '(not in vocab)'
    print(f'{word:20s}  {str(in_vocab):10s}  {str(glove_hit):10s}  {preview}')

word                  in vocab    GloVe hit   vec[:4]
----------------------------------------------------------------------
pytorch               True        True        [ 0.724  0.258  1.298 -0.178]
documentation         True        True        [-0.15  -0.07  -0.276 -0.19 ]
pokemon               True        True        [-0.031 -0.142  0.345  0.166]
reddit                True        True        [-0.001  0.21  -0.327  0.001]
attention             False       False       (not in vocab)
zzzzunknownword       False       False       (not in vocab)


In [8]:
# Structural embedding matrix — each tag should have a distinct non-zero row
print(f'Structural embeddings [{struct_matrix.shape[0]} rows x {struct_matrix.shape[1]} dims]')
print(f'{"idx":4s}  {"tag":12s}  mean    std     first 4 values')
print('-' * 65)

idx_to_tag = {0: '<PAD>'}
idx_to_tag.update({v: k for k, v in config.TAG_TO_IDX.items()})

for i, row in enumerate(struct_matrix):
    tag = idx_to_tag.get(i, '?')
    print(f'{i:4d}  {tag:12s}  {row.mean():+.4f}  {row.std():.4f}  {row[:4].round(4)}')

Structural embeddings [13 rows x 16 dims]
idx   tag           mean    std     first 4 values
-----------------------------------------------------------------
   0  <PAD>         +0.0000  0.0000  [0. 0. 0. 0.]
   1  title         +0.0632  0.2235  [ 0.2884 -0.0093 -0.1028 -0.0232]
   2  meta_desc     -0.0121  0.2811  [ 0.0409 -0.0785  0.1654  0.4147]
   3  h1            -0.0875  0.1921  [-0.0151 -0.0087 -0.2151 -0.3195]
   4  h2            +0.0694  0.3342  [ 0.4612 -0.1892  0.1148  0.5639]
   5  h3            +0.0193  0.1909  [-0.2229  0.0567  0.1283 -0.0678]
   6  h4            +0.0264  0.2616  [ 0.409   0.4672 -0.1399  0.2196]
   7  h5            -0.0707  0.2141  [ 0.3948  0.0026 -0.2093  0.0385]
   8  strong        +0.0384  0.2424  [ 0.2704 -0.1207 -0.3373 -0.2354]
   9  em            -0.0305  0.2224  [ 0.0209 -0.2692  0.0831 -0.4019]
  10  span          +0.1248  0.2411  [0.3937 0.0199 0.1848 0.3237]
  11  p             +0.0413  0.1341  [0.0726 0.1434 0.0531 0.0017]
  12  table      

## 5. Feature matrix — `extract_page`

In [9]:
url, html_path, label = records[4]  # change index to inspect a different page
R, omega, mask = extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M)

k = word_matrix.shape[1]
n = struct_matrix.shape[1]

print(f'Page: {url}  [{label}]')
print()
print(f'R tuples:     {len(R)}')
print(f'omega shape:  {omega.shape}  (expected [{config.M}, {k+n}])')
print(f'mask shape:   {mask.shape}')
print(f'Real tokens:  {mask.sum()} / {config.M}')
print(f'Padded rows:  {(~mask).sum()}')
print()

# Python assertions
assert omega.shape == (config.M, k + n), f'omega shape mismatch: {omega.shape}'
assert mask.shape == (config.M,)
assert mask.dtype == bool
print('Shape assertions passed')

pad_rows = omega[~mask]
assert (pad_rows == 0).all(), 'FAIL: padding rows are not zero'
print('Padding rows are all-zero — OK')

for i, (tok, word_vec, struct_vec) in enumerate(R[:5]):
    assert np.array_equal(omega[i, :k], word_vec),   f'word_vec mismatch at row {i}'
    assert np.array_equal(omega[i, k:], struct_vec), f'struct_vec mismatch at row {i}'
print('Ω_i = [E_i ; S_j] concatenation check — OK')
print()

# Rust correctness
R_rs, omega_rs, mask_rs = rust_extractor.extract_page(
    html_path, token2idx, word_matrix, struct_matrix, config.M
)
assert omega_rs.shape == (config.M, k + n), f'Rust omega shape mismatch: {omega_rs.shape}'
assert np.array_equal(mask, mask_rs), 'Rust mask mismatch'
assert all(t_py == t_rs for (t_py,_,_),(t_rs,_,_) in zip(R, R_rs)), 'Rust token sequence mismatch'
assert (omega_rs[~mask_rs] == 0).all(), 'Rust: non-zero padding rows'
print('Rust extract_page: PASS')

Page: docs.pytorch.org/docs/stable/index.html  [productive]

R tuples:     512
omega shape:  (512, 316)  (expected [512, 316])
mask shape:   (512,)
Real tokens:  512 / 512
Padded rows:  0

Shape assertions passed
Padding rows are all-zero — OK
Ω_i = [E_i ; S_j] concatenation check — OK

Rust extract_page: PASS


In [10]:
# Inspect first 10 R tuples
print(f'First 10 R = (t, e, p) tuples from: {url}\n')
print(f'{"token":20s}  {"word_vec[:3]":38s}  struct_vec[:3]')
print('-' * 85)
for tok, word_vec, struct_vec in R[:10]:
    print(f'{tok:20s}  {str(word_vec[:3].round(4)):38s}  {struct_vec[:3].round(4)}')

First 10 R = (t, e, p) tuples from: docs.pytorch.org/docs/stable/index.html

token                 word_vec[:3]                            struct_vec[:3]
-------------------------------------------------------------------------------------
pytorch               [0.724  0.2582 1.298 ]                  [ 0.2884 -0.0093 -0.1028]
documentation         [-0.1503 -0.0704 -0.2759]               [ 0.2884 -0.0093 -0.1028]
pytorch               [0.724  0.2582 1.298 ]                  [ 0.2884 -0.0093 -0.1028]
2                     [-0.2078 -0.2103  0.2284]               [ 0.2884 -0.0093 -0.1028]
10                    [-0.3736 -0.2026  0.1754]               [ 0.2884 -0.0093 -0.1028]
documentation         [-0.1503 -0.0704 -0.2759]               [ 0.2884 -0.0093 -0.1028]
opens                 [-0.3437 -0.0667  0.0892]               [0.3937 0.0199 0.1848]
in                    [-0.2735 -0.0274 -0.2061]               [0.3937 0.0199 0.1848]
a                     [-0.1813 -0.1594 -0.0779]               

## 6. Timing — single page extraction

In [11]:
import timeit
import rust_extractor, sys                                                                                                                         
print(rust_extractor.__file__)                                                                                                                     
print(sys.executable)  
url, html_path, label = records[4]  # change index to time a different page

N_RUNS = 1000
py_elapsed = timeit.timeit(
    lambda: extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M),
    number=N_RUNS,
)
rs_elapsed = timeit.timeit(
    lambda: rust_extractor.extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M),
    number=N_RUNS,
)

py_ms = (py_elapsed / N_RUNS) * 1000
rs_ms = (rs_elapsed / N_RUNS) * 1000

print(f'Page:    {url}  [{label}]')
print(f'Runs:    {N_RUNS}')
print()
print(f'Python : {py_elapsed:.3f}s total  {py_ms:.3f} ms/call')
print(f'Rust   : {rs_elapsed:.3f}s total  {rs_ms:.3f} ms/call')
print(f'Speedup: {py_ms/rs_ms:.1f}x')

/Users/umair/mlops/models/.venv/lib/python3.13/site-packages/rust_extractor/__init__.py
/Users/umair/mlops/models/.venv/bin/python
Page:    docs.pytorch.org/docs/stable/index.html  [productive]
Runs:    1000

Python : 6.793s total  6.793 ms/call
Rust   : 0.617s total  0.617 ms/call
Speedup: 11.0x


## 7. Full corpus extraction

In [12]:
import timeit

print(f'Extracting features for all {len(records)} pages...\n')
print(f'{"label":12s}  {"url":45s}  py  rs  match')
print('-' * 95)

for url, html_path, label in records:
    R,    omega,    mask    = extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M)
    R_rs, omega_rs, mask_rs = rust_extractor.extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M)

    assert omega.shape    == (config.M, k + n)
    assert omega_rs.shape == (config.M, k + n)
    assert mask.dtype == bool
    assert (omega[~mask] == 0).all(),       f'Non-zero padding in {url}'
    assert (omega_rs[~mask_rs] == 0).all(), f'Rust non-zero padding in {url}'

    tokens_match = np.array_equal(mask, mask_rs) and all(
        tp == tr for (tp,_,_),(tr,_,_) in zip(R, R_rs)
    )
    status = 'OK' if tokens_match else 'MISMATCH'
    print(f'[{label:10s}]  {url:45s}  {mask.sum():3d}  {mask_rs.sum():3d}  {status}')

print()
print('All pages passed')
print()

# Speed comparison across the full corpus
N_RUNS = 10
py_ms = timeit.timeit(
    lambda: [extract_page(hp, token2idx, word_matrix, struct_matrix, config.M)
             for _, hp, _ in records],
    number=N_RUNS,
) / N_RUNS * 1000

rs_ms = timeit.timeit(
    lambda: [rust_extractor.extract_page(hp, token2idx, word_matrix, struct_matrix, config.M)
             for _, hp, _ in records],
    number=N_RUNS,
) / N_RUNS * 1000

print(f'Full corpus ({len(records)} pages), avg over {N_RUNS} runs:')
print(f'  Python : {py_ms:.1f} ms/corpus')
print(f'  Rust   : {rs_ms:.1f} ms/corpus')
print(f'  Speedup: {py_ms/rs_ms:.1f}x')

Extracting features for all 20 pages...

label         url                                            py  rs  match
-----------------------------------------------------------------------------------------------
[waste     ]  pokemon.com/us                                 512  512  OK
[waste     ]  minecraft.net/en-us                            512  512  OK
[waste     ]  reddit.com/r/playboicarti/hot/                 435  435  OK
[waste     ]  reddit.com/r/ChainsawMan/                      512  512  OK
[productive]  docs.pytorch.org/docs/stable/index.html        512  512  OK
[productive]  chatgpt.com/                                    51   51  OK
[skip      ]  youtube.com/                                   512  512  OK
[productive]  developer.apple.com/documentation/swift/       512  512  OK
[productive]  developer.apple.com/documentation/swift/swift-standard-library  512  512  OK
[productive]  rust-lang.org/learn/                           380  380  OK
[productive]  doc.rust-lang.org